# Silver Layer Monitor
Inventory, incoming data pipeline health, and outgoing data status for the LEAGUE_RECORDS.SILVER schema.

In [ ]:
%%sql -r ctx
USE DATABASE LEAGUE_RECORDS;
USE SCHEMA SILVER;

---
## 1. Object Inventory
Take stock of all objects in the Silver schema: tables, views, streams, tasks, and dynamic tables.

In [ ]:
SHOW TABLES IN SCHEMA LEAGUE_RECORDS.SILVER;

In [ ]:
SHOW VIEWS IN SCHEMA LEAGUE_RECORDS.SILVER;

In [ ]:
SHOW TASKS IN SCHEMA LEAGUE_RECORDS.SILVER;

---
## 2. Incoming Data Monitoring
Monitor the pipeline feeding data into silver: task execution history, stream lag, and row counts.

In [ ]:
SELECT
    NAME,
    STATE,
    SCHEDULED_TIME,
    COMPLETED_TIME,
    DATEDIFF('second', SCHEDULED_TIME, COMPLETED_TIME) AS duration_sec,
    ERROR_CODE,
    ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD(
        'hour', -24, CURRENT_TIMESTAMP()
    ),
    RESULT_LIMIT => 100
))
WHERE SCHEMA_NAME = 'SILVER'
ORDER BY NAME, SCHEDULED_TIME DESC;

In [ ]:
SELECT
    NAME AS task_name,
    STATE,
    COUNT(*) AS run_count,
    ROUND(AVG(DATEDIFF('second', SCHEDULED_TIME, COMPLETED_TIME)), 2) AS avg_duration_sec
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD('hour', -24, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 500
))
WHERE SCHEMA_NAME = 'SILVER'
GROUP BY NAME, STATE
ORDER BY NAME, STATE;

In [ ]:
SELECT
    'MATCH_INTERVALS_BRONZE_STM' AS stream_name,
    SYSTEM$STREAM_HAS_DATA('BRONZE.MATCH_INTERVALS_BRONZE_STM') AS has_data
UNION ALL
SELECT 'MATCHES_SUMMARY_BRONZE_STM',
    SYSTEM$STREAM_HAS_DATA('BRONZE.MATCHES_SUMMARY_BRONZE_STM')
UNION ALL
SELECT 'PLAYERS_SUMMARY_BRONZE_STM',
    SYSTEM$STREAM_HAS_DATA('BRONZE.PLAYERS_SUMMARY_BRONZE_STM')
UNION ALL
SELECT 'ITEMS_REF_BRONZE_STM',
    SYSTEM$STREAM_HAS_DATA('BRONZE.ITEMS_REF_BRONZE_STM')
UNION ALL
SELECT 'CHAMPIONS_REF_BRONZE_STM',
    SYSTEM$STREAM_HAS_DATA('BRONZE.CHAMPIONS_REF_BRONZE_STM')
;

In [ ]:
SELECT
    TABLE_NAME,
    ROW_COUNT,
    BYTES,
    ROUND(BYTES / 1024 / 1024, 2) AS size_mb,
    LAST_ALTERED
FROM LEAGUE_RECORDS.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'SILVER'
    AND TABLE_TYPE = 'BASE TABLE'
ORDER BY ROW_COUNT DESC;

---
## 3. Outgoing Data Monitoring
Monitor data flowing out of silver into the gold layer: dynamic table refresh status, row counts, and staleness.

In [ ]:
%%sql -r silver_vs_gold_counts
-- Compare silver vs gold row counts for key relationships
SELECT
    'MATCHES: Silver vs Gold' AS comparison,
    (SELECT COUNT(*) FROM SILVER.MATCHES_SUMMARY_SILVER) AS silver_rows,
    (SELECT COUNT(*) FROM GOLD.MATCH_TEAM_STATS_SUMMARY) AS gold_rows,
    (SELECT COUNT(*) FROM SILVER.MATCHES_SUMMARY_SILVER) -
    (SELECT COUNT(*) FROM GOLD.MATCH_TEAM_STATS_SUMMARY) AS diff
UNION ALL
SELECT
    'PLAYERS: Silver vs Gold',
    (SELECT COUNT(*) FROM SILVER.PLAYERS_SUMMARY_SILVER),
    (SELECT COUNT(*) FROM GOLD.PLAYER_STATS_SUMMARY),
    (SELECT COUNT(*) FROM SILVER.PLAYERS_SUMMARY_SILVER) -
    (SELECT COUNT(*) FROM GOLD.PLAYER_STATS_SUMMARY)
UNION ALL
SELECT
    'CHAMPIONS: Ref vs Overview',
    (SELECT COUNT(*) FROM SILVER.CHAMPIONS_REF_SILVER WHERE CHAMPION_ID != 0),
    (SELECT COUNT(*) FROM GOLD.CHAMPION_OVERVIEW),
    (SELECT COUNT(*) FROM SILVER.CHAMPIONS_REF_SILVER WHERE CHAMPION_ID != 0) -
    (SELECT COUNT(*) FROM GOLD.CHAMPION_OVERVIEW)
;

In [ ]:
# Outgoing data summary
import pandas as pd

refresh_df = dt_refresh_history.to_pandas() if hasattr(dt_refresh_history, 'to_pandas') else dt_refresh_history
lag_df = dt_lag_status.to_pandas() if hasattr(dt_lag_status, 'to_pandas') else dt_lag_status
compare_df = silver_vs_gold_counts.to_pandas() if hasattr(silver_vs_gold_counts, 'to_pandas') else silver_vs_gold_counts

print('='*50)
print('  OUTGOING DATA HEALTH (Silver -> Gold)')
print('='*50)

# DT refresh failures
if len(refresh_df) > 0:
    state_col = 'STATE' if 'STATE' in refresh_df.columns else 'state'
    failures = refresh_df[refresh_df[state_col] != 'SUCCEEDED']
    print(f'\n  DT refreshes (24h):      {len(refresh_df)}')
    print(f'  DT refresh failures:     {len(failures)}')
    if len(failures) > 0:
        print('\n  ✗ FAILED REFRESHES:')
        name_col = 'NAME' if 'NAME' in refresh_df.columns else 'name'
        msg_col = 'STATE_MESSAGE' if 'STATE_MESSAGE' in refresh_df.columns else 'state_message'
        for _, row in failures.head(5).iterrows():
            print(f"    - {row[name_col]}: {row.get(msg_col, 'N/A')}")
    else:
        print('  ✓ All refreshes succeeded')
else:
    print('\n  No refresh history in last 24h')

# Scheduling state
name_col = 'name' if 'name' in lag_df.columns else 'NAME'
state_col = 'scheduling_state' if 'scheduling_state' in lag_df.columns else 'SCHEDULING_STATE'
lag_col = 'target_lag' if 'target_lag' in lag_df.columns else 'TARGET_LAG'

print(f'\n  Dynamic Table Scheduling:')
for _, row in lag_df.iterrows():
    state = row.get(state_col, 'UNKNOWN')
    icon = '✓' if state in ('ACTIVE', 'RUNNING') else '⚠'
    print(f"    {icon} {str(row[name_col]):35s} {state} (lag: {row.get(lag_col, '?')})")

# Row count comparison
print(f'\n  Silver -> Gold Row Counts:')
for _, row in compare_df.iterrows():
    diff = int(row['DIFF'])
    icon = '✓' if diff == 0 else '⚠'
    print(f"    {icon} {row['COMPARISON']:30s} silver={row['SILVER_ROWS']}  gold={row['GOLD_ROWS']}  diff={diff}")

print('\n' + '='*50)